In [1]:
import json
import pandas as pd
import requests
import time
import re
import multiprocessing
import math
from transformers import BertModel, BertConfig, BertTokenizer
import torch,gc
from collections import Counter

### Common Function

In [2]:
# 定义一个函数来比较两个字符串的相似度
def compare_similarity(row, df, column_name):
    # 使用process提取相似度最高的记录
    highest = process.extractOne(row[column_name], df[column_name])
    # 返回相似度
    return highest[1]

### Handle Data

In [3]:
problem_data = []

# 打开JSON文件
with open('/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/problem.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        problem = json.loads(line)
        Id = problem["problem_id"]
        exercise_id = problem["exercise_id"]
        language = problem["language"]
        title = problem["title"]
        content = problem["content"] 
        option = problem["option"]
        answer = problem["answer"]
        score = problem["score"]
        Type = problem["type"]
        typetext = problem["typetext"]
        problem_data.append([Id,exercise_id,language,title,content,option,answer,score,Type,typetext])


In [4]:
df_problem = pd.DataFrame(problem_data,columns=["id","exercise_id","language","title","content","option","answer","score","type","typetext"])

In [15]:
df_problem = df_problem.drop('answer', axis=1)
df_problem = df_problem.drop('title', axis=1)
df_problem = df_problem.drop('option', axis=1)

In [18]:
df_problem = df_problem.drop('exercise_id', axis=1)

In [19]:
df_problem["typetext"].value_counts()

typetext
单选题    1421954
判断题     505433
多选题     373565
填空题     112871
主观题      37449
投票题       2869
编程题        281
Name: count, dtype: int64

In [22]:
### 清洗掉 content 中的序号
content = df_problem['content']
pattern = r'^\s*\d+[\u3001、]'
# 使用str.replace方法替换匹配到的部分为空字符串
content_cleaned = content.str.replace(pattern, '', regex=True)
df_problem['content'] = content_cleaned

In [23]:
### 清洗掉 content 中是文件的数据
file_extensions = ['.txt', '.jpg', '.mp4', '.wav', '.png', '.pdf', '.docx', '.xls','.doc','.pptx','.ppt']
pattern1 = '|'.join(file_extensions).replace('.', '\\.')
df_problem = df_problem[~df_problem['content'].str.contains(pattern1, regex=True)]

In [24]:
df_problem

,id,language,content,score,type,typetext
0,1730,Chinese,《资治通鉴》卷1记载：智宣子将以瑶为后，智果曰：“……瑶之贤于人者五，其不逮者一也。美鬓长大...,1.0,1,单选题
1,1731,Chinese,《资治通鉴》是一部____史书。,1.0,1,单选题
2,1732,Chinese,《资治通鉴》原名____，后由____赐名“资治通鉴”。,1.0,1,单选题
3,1733,Chinese,“三家分晋”中“三家”具体指：,1.0,1,单选题
4,1734,Chinese,智伯联合韩、魏的军队攻打赵氏时，赵襄子选择退守的阵地是：,1.0,1,单选题
...,...,...,...,...,...,...
2454417,8431749,English,"Among the following works, which is not Swift'...",NaN,1,单选题
2454418,8431750,English,"From the dialogue, we know that Swift's best k...",NaN,1,单选题
2454419,8431751,English,"In this novel, Gulliver travelled to ( ).",NaN,1,单选题
2454420,8431752,English,"From the conversation, we know that Eve really...",NaN,1,单选题


In [25]:
df_problem = df_problem.drop_duplicates(subset='content', keep='last')

In [26]:
df_problem

,id,language,content,score,type,typetext
3601,13816,Chinese,中国共产党从成立之日起，就确立了（ ）的远大理想，始终团结带领中国人民朝着这个伟大目标前进。,1.0,1,单选题
3602,13817,Chinese,走好新时代的长征路，大学生要不断增强中国特色社会主义（ ），自觉做共产主义远大理想和中国特色...,1.0,1,单选题
3614,13829,Chinese,国家安全问题事关国家安危和民族存亡，大学生要增强国家安全意识，以下做法错误的是（ ）,1.0,1,单选题
3617,13832,Chinese,对于社会主义核心价值观的自信，来自于（ ）,2.0,2,多选题
3620,13835,Chinese,党的十八大提出，要倡导（ ），倡导（ ），倡导（ ），积极培育和践行社会主义核心价值观。,1.0,1,单选题
...,...,...,...,...,...,...
2454417,8431749,English,"Among the following works, which is not Swift'...",NaN,1,单选题
2454418,8431750,English,"From the dialogue, we know that Swift's best k...",NaN,1,单选题
2454419,8431751,English,"In this novel, Gulliver travelled to ( ).",NaN,1,单选题
2454420,8431752,English,"From the conversation, we know that Eve really...",NaN,1,单选题


In [27]:
### 清洗掉 socre 中值为NaN的行
# df_problem.dropna(subset=['score'], inplace=True)

### 在清洗掉content 部分数据
df_problem = df_problem.loc[~((df_problem['language'] == 'Chinese') & (df_problem['content'].str.contains(r'\[填空1\]')))]

### 在清洗掉content 部分数据
df_problem = df_problem.loc[~((df_problem['language'] == 'English') & (df_problem['content'].str.contains(r'\$')))]


In [28]:
### 清除<10  >128的数据
df_problem['str_len'] = df_problem['content'].str.len()

In [29]:
# 删除str_len列中小于10或大于128的行
df_problem = df_problem[(df_problem['str_len'] >= 10) & (df_problem['str_len'] <= 128)]

In [30]:
df_problem = df_problem.drop('str_len', axis=1)

In [31]:
df_problem

,id,language,content,score,type,typetext
3601,13816,Chinese,中国共产党从成立之日起，就确立了（ ）的远大理想，始终团结带领中国人民朝着这个伟大目标前进。,1.0,1,单选题
3602,13817,Chinese,走好新时代的长征路，大学生要不断增强中国特色社会主义（ ），自觉做共产主义远大理想和中国特色...,1.0,1,单选题
3614,13829,Chinese,国家安全问题事关国家安危和民族存亡，大学生要增强国家安全意识，以下做法错误的是（ ）,1.0,1,单选题
3617,13832,Chinese,对于社会主义核心价值观的自信，来自于（ ）,2.0,2,多选题
3620,13835,Chinese,党的十八大提出，要倡导（ ），倡导（ ），倡导（ ），积极培育和践行社会主义核心价值观。,1.0,1,单选题
...,...,...,...,...,...,...
2454417,8431749,English,"Among the following works, which is not Swift'...",NaN,1,单选题
2454418,8431750,English,"From the dialogue, we know that Swift's best k...",NaN,1,单选题
2454419,8431751,English,"In this novel, Gulliver travelled to ( ).",NaN,1,单选题
2454420,8431752,English,"From the conversation, we know that Eve really...",NaN,1,单选题


In [32]:
df_problem.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/csv/problem/problem_processed.csv', index=False)

In [33]:
df_problem_content = df_problem["content"]
df_problem_content_list = df_problem_content.tolist()

### 使用 bert-base-multilingual-cased 模型

In [34]:
def bert_define(path="/remote-home/cs_acmis_wsf/ai4dingo/model/bert_multi_pretrained"):
    # 加载bert的tokenizer分词
    tokenizer = BertTokenizer.from_pretrained(path)
    # 加载预训练模型
    model_config = BertConfig.from_pretrained(path)
    model = BertModel.from_pretrained(path, config=model_config)
    model = model.cuda()
    return tokenizer, model

In [35]:
tokenizer,model = bert_define()

/root/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. P

In [36]:
# 构成一个小batch
batch_token, batch_segment, batch_mask = list(), list(), list()

In [37]:
text = df_problem["content"].tolist()

In [38]:
max_len = max(len(t) for t in text)

In [41]:
for t in text:
    # text 作 tokenizer 分词
    token = tokenizer.tokenize(t)
    token = ['[CLS]'] + token + ['[SEP]']
    token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

    # 加padding补齐及segment、mask
    padding = [0] * (max_len - len(token_id))
    mask = [1] * len(token_id) + padding
    segment = [0] * len(token_id) + padding
    token_id = token_id + padding

    batch_token.append(token_id)
    batch_segment.append(segment)
    batch_mask.append(mask)


#### 清除 dim > max_len的数据

In [42]:
dim  = [len(d) for d in batch_token]

In [43]:
max(dim)

130

In [44]:
err_dim = []
for i in range(len(dim)):
    if dim[i] != max_len:
        err_dim.append(i)

In [45]:
batch_token_new  = []
for bt in batch_token:
    if len(bt) == max_len:
        batch_token_new.append(bt)
        
        
batch_segment_new = []
for bs in batch_segment:
    if len(bs) == max_len:
        batch_segment_new.append(bs)
        
batch_mask_new = []
for bm in batch_mask:
    if len(bm) == max_len:
        batch_mask_new.append(bm)

In [46]:
df_problem = df_problem.reset_index(drop=True)
df_problem = df_problem.drop(err_dim)
df_problem = df_problem.reset_index(drop=True)

In [47]:
vector_count = min([len(df_problem),len(batch_mask),len(batch_segment),len(batch_token)])

In [48]:
vector_count

211618

#### 计算最近best_n, best_batch_size, best_batch

In [49]:
def closest_batch_size(vector_count, min_n=6, max_n=12):
    best_n = min_n
    best_batch_size = 2**best_n
    best_batch = vector_count // best_batch_size
    min_difference = abs(vector_count - best_batch * best_batch_size)

    for n in range(min_n + 1, max_n + 1):
        batch_size = 2**n
        batch = vector_count // batch_size
        difference = abs(vector_count - batch * batch_size)

        # 检查当前batch_size是否比之前的更接近vector_count
        if difference < min_difference:
            best_n = n
            best_batch_size = batch_size
            best_batch = batch
            min_difference = difference

    return best_n, best_batch_size, best_batch

In [50]:
n, batch_size, batch = closest_batch_size(vector_count)
print(f"vector_count: {vector_count}, n: {n}, batch_size: {batch_size}, batch: {batch}")

vector_count: 211618, n: 6, batch_size: 64, batch: 3306


#### 计算 max_vector_count

In [51]:
max_vector_count = 2**n * batch

In [52]:
df_problem = df_problem.head(max_vector_count)

In [54]:
df_problem.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/csv/problem/problem_max_vector.csv', index=False)

In [55]:
def split_dataframe(df, n_parts, rows_per_part):
    # 计算总行数
    total_rows = len(df)
    # 计算前n-1份的总行数
    rows_for_first_n_minus_one_parts = (n_parts - 1) * rows_per_part
    # 确定是否需要额外的部分来存储剩余的数据
    extra_part_needed = total_rows > rows_for_first_n_minus_one_parts

    # 创建一个包含所有分片的列表
    dfs = []

    # 分配前n-1份
    for i in range(n_parts - 1):
        start = i * rows_per_part
        end = start + rows_per_part
        dfs.append(df.iloc[start:end])

    # 如果需要，分配最后一部分
    if extra_part_needed:
        start = (n_parts - 1) * rows_per_part
        dfs.append(df.iloc[start:])

    return dfs


In [56]:
n = 7
rows_per_part = 34816  # 每份的行数
split_dfs_problem = split_dataframe(df_problem, n,rows_per_part)

In [57]:
df_problem

,id,language,content,score,type,typetext
0,13816,Chinese,中国共产党从成立之日起，就确立了（ ）的远大理想，始终团结带领中国人民朝着这个伟大目标前进。,1.0,1,单选题
1,13817,Chinese,走好新时代的长征路，大学生要不断增强中国特色社会主义（ ），自觉做共产主义远大理想和中国特色...,1.0,1,单选题
2,13829,Chinese,国家安全问题事关国家安危和民族存亡，大学生要增强国家安全意识，以下做法错误的是（ ）,1.0,1,单选题
3,13832,Chinese,对于社会主义核心价值观的自信，来自于（ ）,2.0,2,多选题
4,13835,Chinese,党的十八大提出，要倡导（ ），倡导（ ），倡导（ ），积极培育和践行社会主义核心价值观。,1.0,1,单选题
...,...,...,...,...,...,...
211579,8431708,English,Being lost in a strange place after dark was a...,NaN,1,单选题
211580,8431709,English,The table is not ( ) wide for our purpose.,NaN,1,单选题
211581,8431710,English,There is no ( ) charge for children under 12.,NaN,1,单选题
211582,8431711,English,"To my absolute ( ), the scheme was a huge succ...",NaN,1,单选题


### 循环处理

In [58]:
batch_size_num = 2048

for i in range(len(split_dfs_problem)):
    df = split_dfs_problem[i]
    print(f'正在读取第 {i+1} 个DF')
    text = df["content"].tolist()
    
    print(f'初始化batch数据： {i+1}')
    batch_token, batch_segment, batch_mask = list(), list(), list()
    for t in text:
        # text 作 tokenizer 分词
        token = tokenizer.tokenize(str(t))
        token = ['[CLS]'] + token + ['[SEP]']
        token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

        # 加padding补齐及segment、mask
        padding = [0] * (max_len - len(token_id))
        mask = [1] * len(token_id) + padding
        segment = [0] * len(token_id) + padding
        token_id = token_id + padding

        batch_token.append(token_id)
        batch_segment.append(segment)
        batch_mask.append(mask)
        
    print(f'batch数据 组装完毕 ：{i+1}')
   
    batch_tensor_token = torch.tensor(batch_token)
    batch_tensor_segment = torch.tensor(batch_segment)
    batch_tensor_mask = torch.tensor(batch_mask)

    batch_len = len(batch_tensor_token)
    
    if torch.cuda.is_available():
        batch_tensor_token = batch_tensor_token.to('cuda:0')
        batch_tensor_segment = batch_tensor_segment.to('cuda:0')
        batch_tensor_mask = batch_tensor_mask.to('cuda:0')
    
    print(f'batch_tensor数据 组装完毕 ：{i+1},开始分块处理')
    
    batch_tensor_token_chunked = [batch_tensor_token[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_segment_chunked = [batch_tensor_segment[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_mask_chunked = [batch_tensor_mask[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    print(f'分块处理完毕数据完毕，每块的长度为:{batch_size_num},一共有 {len(batch_tensor_token_chunked)} 块， 开始调用模型')
    
    gc.collect()
    torch.cuda.empty_cache()
    text_vector = []
    
    try:
        for k in range(len(batch_tensor_token_chunked)):
            with torch.no_grad():
                print(f'正在处理 {i+1} ----- {k+1}')
                outputs = model(batch_tensor_token_chunked[k], token_type_ids=batch_tensor_segment_chunked[k], attention_mask=batch_tensor_mask_chunked[k])
                outputs = outputs[0][:, 0, :]  # 取cls向量
                for o in range(outputs.shape[0]):
                    text_vector.append(outputs[o])
        df["content_vector"] = [ tv.tolist() for tv in  text_vector] 
        df.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/csv/problem/part_{i+1}.csv', index=False)
        print(f"文件写入完毕 {i+1}")
                    
    except Exception as e:
        print(f"出现异常 {i+1}: {e}")
        continue
            
    
    print('============================')
    print()

正在读取第 1 个DF
初始化batch数据： 1
batch数据 组装完毕 ：1
batch_tensor数据 组装完毕 ：1,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 1 ----- 1
正在处理 1 ----- 2
正在处理 1 ----- 3
正在处理 1 ----- 4
正在处理 1 ----- 5
正在处理 1 ----- 6
正在处理 1 ----- 7
正在处理 1 ----- 8
正在处理 1 ----- 9
正在处理 1 ----- 10
正在处理 1 ----- 11
正在处理 1 ----- 12
正在处理 1 ----- 13
正在处理 1 ----- 14
正在处理 1 ----- 15
正在处理 1 ----- 16
正在处理 1 ----- 17


/tmp/ipykernel_175250/16065817.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content_vector"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 1

正在读取第 2 个DF
初始化batch数据： 2
batch数据 组装完毕 ：2
batch_tensor数据 组装完毕 ：2,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 2 ----- 1
正在处理 2 ----- 2
正在处理 2 ----- 3
正在处理 2 ----- 4
正在处理 2 ----- 5
正在处理 2 ----- 6
正在处理 2 ----- 7
正在处理 2 ----- 8
正在处理 2 ----- 9
正在处理 2 ----- 10
正在处理 2 ----- 11
正在处理 2 ----- 12
正在处理 2 ----- 13
正在处理 2 ----- 14
正在处理 2 ----- 15
正在处理 2 ----- 16
正在处理 2 ----- 17


/tmp/ipykernel_175250/16065817.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content_vector"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 2

正在读取第 3 个DF
初始化batch数据： 3
batch数据 组装完毕 ：3
batch_tensor数据 组装完毕 ：3,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 3 ----- 1
正在处理 3 ----- 2
正在处理 3 ----- 3
正在处理 3 ----- 4
正在处理 3 ----- 5
正在处理 3 ----- 6
正在处理 3 ----- 7
正在处理 3 ----- 8
正在处理 3 ----- 9
正在处理 3 ----- 10
正在处理 3 ----- 11
正在处理 3 ----- 12
正在处理 3 ----- 13
正在处理 3 ----- 14
正在处理 3 ----- 15
正在处理 3 ----- 16
正在处理 3 ----- 17


...

文件写入完毕 5

正在读取第 6 个DF
初始化batch数据： 6
batch数据 组装完毕 ：6
batch_tensor数据 组装完毕 ：6,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 6 ----- 1
正在处理 6 ----- 2
正在处理 6 ----- 3
正在处理 6 ----- 4
正在处理 6 ----- 5
正在处理 6 ----- 6
正在处理 6 ----- 7
正在处理 6 ----- 8
正在处理 6 ----- 9
正在处理 6 ----- 10
正在处理 6 ----- 11
正在处理 6 ----- 12
正在处理 6 ----- 13
正在处理 6 ----- 14
正在处理 6 ----- 15
正在处理 6 ----- 16
正在处理 6 ----- 17


/tmp/ipykernel_175250/16065817.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content_vector"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 6

正在读取第 7 个DF
初始化batch数据： 7
batch数据 组装完毕 ：7
batch_tensor数据 组装完毕 ：7,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 2 块， 开始调用模型
正在处理 7 ----- 1
正在处理 7 ----- 2


/tmp/ipykernel_175250/16065817.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content_vector"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 7



In [59]:
base_directory = '/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/csv/problem'
base_filename = 'part_'
file_extension = '.csv'

In [61]:
import os

In [62]:
dfs = []

# 使用循环读取part_1.csv到part_40.csv文件
for i in range(1, 8):  # 从1到40，包含40
    file_name = f"{base_filename}{i}{file_extension}"  # 构建文件名
    file_path = os.path.join(base_directory, file_name)  # 构建完整的文件路径
    df = pd.read_csv(file_path)  # 读取CSV文件
    dfs.append(df)  # 将DataFrame添加到列表中

In [63]:
df = pd.concat(dfs, ignore_index=True)

In [64]:
df

,id,language,content,score,type,typetext,content_vector
0,13816,Chinese,中国共产党从成立之日起，就确立了（ ）的远大理想，始终团结带领中国人民朝着这个伟大目标前进。,1.0,1,单选题,"[0.3620103895664215, 0.4917919635772705, 0.001..."
1,13817,Chinese,走好新时代的长征路，大学生要不断增强中国特色社会主义（ ），自觉做共产主义远大理想和中国特色...,1.0,1,单选题,"[0.307481974363327, 0.13994258642196655, 0.536..."
2,13829,Chinese,国家安全问题事关国家安危和民族存亡，大学生要增强国家安全意识，以下做法错误的是（ ）,1.0,1,单选题,"[-0.15822555124759674, -0.12112200260162354, -..."
3,13832,Chinese,对于社会主义核心价值观的自信，来自于（ ）,2.0,2,多选题,"[0.06346714496612549, -0.10366339236497879, 0...."
4,13835,Chinese,党的十八大提出，要倡导（ ），倡导（ ），倡导（ ），积极培育和践行社会主义核心价值观。,1.0,1,单选题,"[0.07375973463058472, 0.32700297236442566, -0...."
...,...,...,...,...,...,...,...
211585,8431708,English,Being lost in a strange place after dark was a...,NaN,1,单选题,"[0.2641059458255768, 0.043390434235334396, -0...."
211586,8431709,English,The table is not ( ) wide for our purpose.,NaN,1,单选题,"[-0.031159035861492157, 0.005675750784575939, ..."
211587,8431710,English,There is no ( ) charge for children under 12.,NaN,1,单选题,"[-0.16491010785102844, -0.3057374358177185, 0...."
211588,8431711,English,"To my absolute ( ), the scheme was a huge succ...",NaN,1,单选题,"[0.10606086999177933, 0.007934344001114368, -0..."


In [65]:
df = df.rename(columns={'content_vector': 'feature'})

In [66]:
empty_feature_indices = df[df['feature'].isna()].index

In [67]:
empty_feature_indices

Index([21416, 21417, 21425, 21426, 21427, 21438, 21439, 21440, 87469, 87470], dtype='int64')

In [68]:
row_data = df.loc[21416]

In [69]:
row_data

id              3850935
language        Chinese
content     有限精准减压不牵拉脊髓
score               NaN
type                NaN
typetext            NaN
feature             NaN
Name: 21416, dtype: object

In [70]:
df.drop(empty_feature_indices, axis=0, inplace=True)

In [71]:
df = df.reset_index(drop=True)

In [72]:
df

,id,language,content,score,type,typetext,feature
0,13816,Chinese,中国共产党从成立之日起，就确立了（ ）的远大理想，始终团结带领中国人民朝着这个伟大目标前进。,1.0,1,单选题,"[0.3620103895664215, 0.4917919635772705, 0.001..."
1,13817,Chinese,走好新时代的长征路，大学生要不断增强中国特色社会主义（ ），自觉做共产主义远大理想和中国特色...,1.0,1,单选题,"[0.307481974363327, 0.13994258642196655, 0.536..."
2,13829,Chinese,国家安全问题事关国家安危和民族存亡，大学生要增强国家安全意识，以下做法错误的是（ ）,1.0,1,单选题,"[-0.15822555124759674, -0.12112200260162354, -..."
3,13832,Chinese,对于社会主义核心价值观的自信，来自于（ ）,2.0,2,多选题,"[0.06346714496612549, -0.10366339236497879, 0...."
4,13835,Chinese,党的十八大提出，要倡导（ ），倡导（ ），倡导（ ），积极培育和践行社会主义核心价值观。,1.0,1,单选题,"[0.07375973463058472, 0.32700297236442566, -0...."
...,...,...,...,...,...,...,...
211575,8431708,English,Being lost in a strange place after dark was a...,NaN,1,单选题,"[0.2641059458255768, 0.043390434235334396, -0...."
211576,8431709,English,The table is not ( ) wide for our purpose.,NaN,1,单选题,"[-0.031159035861492157, 0.005675750784575939, ..."
211577,8431710,English,There is no ( ) charge for children under 12.,NaN,1,单选题,"[-0.16491010785102844, -0.3057374358177185, 0...."
211578,8431711,English,"To my absolute ( ), the scheme was a huge succ...",NaN,1,单选题,"[0.10606086999177933, 0.007934344001114368, -0..."


In [73]:
df_cleaned = df.dropna()

In [74]:
df_cleaned

,id,language,content,score,type,typetext,feature
0,13816,Chinese,中国共产党从成立之日起，就确立了（ ）的远大理想，始终团结带领中国人民朝着这个伟大目标前进。,1.0,1,单选题,"[0.3620103895664215, 0.4917919635772705, 0.001..."
1,13817,Chinese,走好新时代的长征路，大学生要不断增强中国特色社会主义（ ），自觉做共产主义远大理想和中国特色...,1.0,1,单选题,"[0.307481974363327, 0.13994258642196655, 0.536..."
2,13829,Chinese,国家安全问题事关国家安危和民族存亡，大学生要增强国家安全意识，以下做法错误的是（ ）,1.0,1,单选题,"[-0.15822555124759674, -0.12112200260162354, -..."
3,13832,Chinese,对于社会主义核心价值观的自信，来自于（ ）,2.0,2,多选题,"[0.06346714496612549, -0.10366339236497879, 0...."
4,13835,Chinese,党的十八大提出，要倡导（ ），倡导（ ），倡导（ ），积极培育和践行社会主义核心价值观。,1.0,1,单选题,"[0.07375973463058472, 0.32700297236442566, -0...."
...,...,...,...,...,...,...,...
210242,8420963,English,The endorsor of of Insurance Policy is Shangha...,2.0,6,判断题,"[0.1462920755147934, 0.26734307408332825, 0.09..."
210243,8420964,English,CREDIT NUMBER AND NAME OF ISSUING BANK are no ...,2.0,6,判断题,"[0.09381052106618881, 0.00038472795858979225, ..."
210244,8420965,English,The Bill of Lading must be endorsed in blank b...,2.0,6,判断题,"[0.08344624936580658, 0.11163408309221268, -0...."
210245,8420966,English,The goods description of goods in the Bill of ...,2.0,6,判断题,"[0.3675398528575897, -0.18399843573570251, 0.3..."


In [75]:
df_cleaned = df_cleaned.reset_index(drop=True)

In [77]:
df_cleaned.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/csv/problem/problem.csv', index=False)

In [100]:
# 确保 'score' 列是数值类型  
# df_cleaned['type'] = pd.to_numeric(df_cleaned['type'], errors='coerce')  
  
# 计算 value_counts  
value_counts = df_cleaned["typetext"].value_counts()  

# 过滤掉计数小于10的数据，并将结果转换为DataFrame  
filtered_counts = value_counts[value_counts >= 10].sort_index()  
  
# 重命名索引为'name'，并将计数转换为字符串格式  
result_df = pd.DataFrame({'name': filtered_counts.index, 'num': filtered_counts.values.astype(str)})  
  
# 转换为字典列表，每个字典代表一个记录  
formatted_result = result_df.to_dict('records')  
  
# 输出结果  
print(formatted_result)

[{'name': '主观题', 'num': '3726'}, {'name': '判断题', 'num': '32658'}, {'name': '单选题', 'num': '69088'}, {'name': '填空题', 'num': '569'}, {'name': '多选题', 'num': '19120'}, {'name': '投票题', 'num': '218'}]
